# Renormalization Identity Verification — Read-Only Summary

> **Read-only notebook**: loads and displays `results/exp_r27/identity_check.json` only; it performs no computation and writes nothing.
> Artifacts are generated by `026_exp_r27_renorm_identity.py` (data source = frozen exp0 artifacts seed_42/baseline/fold1; purely deterministic algebraic verification, no randomness, CPU only).

## What This Experiment Addresses

A reviewer raised the concern that "multiplicative correction + per-ITL3 renormalization" might introduce an opaque double-normalization artifact. This experiment responds directly with two **algebraic identities** plus one **decomposition**:

1. **Identity ① (renormalization = a positive per-ITL3 scalar)**: $d^{mult}_i = D_r \cdot \dfrac{b_i f_i}{\sum_j b_j f_j}$, so $d^{mult}_i/(b_i f_i) = \lambda_r > 0$ is constant within a region — the relative ratio between any pair of agents within a region is **strictly unchanged** by the correction; renormalization applies only a positive per-region rescaling and never reorders the within-region shape.
2. **Identity ② (multiplicative correction + renorm ≡ a single softmax(logits + log f))**: Solving for $w_{sa}$ from `gnn_demand = w_sa · D_r` (star-shaped graph: each agent belongs to exactly one ITL3 source), and setting $z = \log w_{sa}$, then $\mathrm{softmax}_r(z + \log f)\cdot D_r$ is componentwise identical to "multiply by $f$, then renormalize" — the post-hoc correction is **equivalent to injecting $\log f$ as a single prior directly into the softmax logits**, adding no extra degrees of freedom.
3. **Decomposition ③ (the RMSE increase of the no-renorm arm = scale term + shape term)**: $d^{nr}_i = b_i f_i = c_r\, d^{mult}_i$ (with $c_r = \sum_j b_j f_j / D_r$), i.e. no-renorm and mult+renorm have **identical shape**, differing only by a per-region scalar. The RMSE degradation of the no-renorm arm is therefore entirely attributable to regional-total mismatch (the scale term); the shape term is zero up to floating-point precision.

**Precondition assertions**: min(f) > 0; the renormalization domain equals the softmax domain (unique star-shaped membership + within-group Σw ≈ 1); the uniform-fallback branch fires 0 times; ITL3 coverage = 100%. All expected/actual/status values are recorded to disk.

In [ ]:
# Load artifact JSON (read-only)
import json
from pathlib import Path

import pandas as pd

EXP_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
JSON_PATH = EXP_DIR / 'results' / 'exp_r27' / 'identity_check.json'
with open(JSON_PATH, encoding='utf-8') as f:
    R = json.load(f)

print('Artifact: ', JSON_PATH)
print('Data source:', R['meta']['source'])
print('Config: seed={seed} / {config} / {fold}, base column={base_col}, {n_locations} regions x arms {arms}'.format(**R['meta']))
print('Generated at:', R['meta']['timestamp'], ' numpy', R['meta']['numpy'])

## 1. Precondition Assertions (four checks: expected / actual / status)

The preconditions for the identities to hold must be satisfied first; otherwise the numerical comparisons that follow are not meaningful.

In [ ]:
pre_rows = []
for key, item in R['preconditions'].items():
    pre_rows.append({
        'Precondition': key,
        'Description': item['description'],
        'expected': json.dumps(item['expected'], ensure_ascii=False),
        'actual': json.dumps(item['actual'], ensure_ascii=False),
        'status': item['status'],
    })
pre_df = pd.DataFrame(pre_rows)
with pd.option_context('display.max_colwidth', 120):
    display(pre_df)

print('Diagnostic (not a formal precondition): number of exactly-zero back-solved weights =', R['diagnostics']['zero_weight_count'],
      '(a float32 underflow artifact; zero weights are 0 along both paths and do not break the identities)')

**Note on the convergence criterion for the within-group sum check**: the absolute value of $|\Sigma w - 1|$ within a group grows linearly with group size $n$ (float32 scatter-softmax denominator accumulated sequentially); the largest group (TLC14, n=37,811) has an absolute deviation of about 1.1e-4. The check therefore uses an error-model-normalized ratio $|\Sigma w-1|/(n\cdot\varepsilon_{32}) < 1$ (a hard upper bound for sequential summation); the observed maximum ratio is ≈0.045, more than 3 orders of magnitude below the $O(1)$ deviation that would indicate a genuine mismatch between the renormalization domain and the softmax domain — ample discriminating power.

## 2. Maximum Deviation for Each Identity Criterion (all 16 regions x 3 factor arms)

Tolerance specification: conservation relative < 1e-12; Identity ① proportionality-constant relative < 1e-12 (pure float64 path); Identity ② large-component (> 1e-12·D_r) relative < 1e-6 [tolerance margin arising from the float32 storage/back-solve path], small-component absolute < 1e-12·D_r; Decomposition ③ shape-term relative < 1e-12 (not exactly zero).

In [ ]:
dev_df = pd.DataFrame(R['max_deviations']).T
num_cols = ['id1_ratio_spread_rel_max', 'conservation_rel_max', 'conservation_recon_rel_max',
            'id2_rel_max_large', 'id2_abs_frac_max_small', 'shape_rel_max']
display(dev_df[num_cols].map(lambda v: f'{v:.3e}')
        .join(dev_df[['id1_status', 'conservation_status', 'id2_status', 'shape_status']]))

print('Tolerances:', json.dumps(R['tolerances'], indent=2))
print()
ov = R['overall']
print(f"Overall max deviation across all arms:  ① ratio {ov['id1_ratio_spread_rel_max']:.3e}   conservation {ov['conservation_rel_max']:.3e}"
      f"   ② large component {ov['id2_rel_max_large']:.3e}   ② small component {ov['id2_abs_frac_max_small']:.3e}"
      f"   ③ shape term {ov['shape_rel_max']:.3e}")

## 3. Decomposition Table: RMSE Increase of the No-Renorm Arm = Scale Term + Shape Term

`rmse_scale_only` = a counterfactual obtained by multiplying the mult+renorm result by $c_r$ per region (applying only the scale mismatch while keeping the renorm shape). If the shape term is zero, then `rmse_scale_only ≈ rmse_no_renorm`, i.e. all of the no-renorm degradation comes from regional-total mismatch.

In [ ]:
dec = pd.DataFrame(R['decomposition_table'])

# Aggregate by arm (mean / extremes across 16 regions)
summary = dec.groupby('arm').agg(
    rmse_mult_mean=('rmse_mult_renorm', 'mean'),
    rmse_nr_mean=('rmse_no_renorm', 'mean'),
    delta_total_mean=('delta_total', 'mean'),
    delta_total_min=('delta_total', 'min'),
    delta_total_max=('delta_total', 'max'),
    delta_scale_mean=('delta_scale', 'mean'),
    delta_shape_abs_max=('delta_shape', lambda s: s.abs().max()),
    delta_shape_rel_max=('delta_shape_rel', 'max'),
).loc[['N', 'P', 'NP']]
display(summary)

# Per-region detail (NP arm)
cols = ['location', 'rmse_baseline', 'rmse_mult_renorm', 'rmse_no_renorm',
        'rmse_scale_only', 'delta_total', 'delta_scale', 'delta_shape_rel']
display(dec[dec['arm'] == 'NP'][cols].round(6).reset_index(drop=True))

In [ ]:
# Magnitude of scale mismatch (distribution of c_r) - the sole source of no-renorm degradation
sf_rows = []
for loc, arms in R['scale_factors_per_itl3'].items():
    for arm, d in arms.items():
        for itl3, c in d.items():
            sf_rows.append({'location': loc, 'arm': arm, 'ITL3': itl3, 'c_r': c})
sf = pd.DataFrame(sf_rows)
display(sf.groupby('arm')['c_r'].agg(['min', 'median', 'max', 'count']).loc[['N', 'P', 'NP']])
print('c_r = within-region Σ(base·f)/D_r. The farther from 1, the larger the regional-total mismatch; '
      'the no-renorm arm leaves this mismatch directly on the demand values.')

## 4. Conclusion (generated programmatically from the recorded numbers)

In [ ]:
# All conclusion lines are generated programmatically from the recorded numbers; none are hand-written
ov = R['overall']
tol = R['tolerances']

checks = [
    ('Precondition assertions (four checks)', ov['all_preconditions_ok']),
    (f"Identity ① (renorm = positive per-ITL3 scalar, relative deviation {ov['id1_ratio_spread_rel_max']:.1e} < {tol['id1_rel']:g})",
     ov['id1_ratio_spread_rel_max'] < tol['id1_rel']),
    (f"Conservation (max across both paths {max(ov['conservation_rel_max'], ov['conservation_recon_rel_max']):.1e} < {tol['conservation_rel']:g})",
     max(ov['conservation_rel_max'], ov['conservation_recon_rel_max']) < tol['conservation_rel']),
    (f"Identity ② (mult+renorm ≡ softmax(logits+log f), large-component relative {ov['id2_rel_max_large']:.1e} < {tol['id2_rel_large']:g}, "
     f"small-component absolute/D_r {ov['id2_abs_frac_max_small']:.1e} < {tol['id2_abs_frac_small']:g})",
     ov['id2_rel_max_large'] < tol['id2_rel_large'] and ov['id2_abs_frac_max_small'] < tol['id2_abs_frac_small']),
    (f"Decomposition ③ (shape-term relative {ov['shape_rel_max']:.1e} < {tol['shape_rel']:g} — the no-renorm increase is entirely a scale term)",
     ov['shape_rel_max'] < tol['shape_rel']),
]
for label, ok in checks:
    print(('[PASS] ' if ok else '[FAIL] ') + label)

if all(ok for _, ok in checks):
    print('\n=> Conclusion holds: renormalization is a positive per-ITL3 scalar rescaling (it does not reorder the within-region shape),')
    print('   the multiplicative post-hoc correction is overall equivalent to injecting log f as a single prior into the softmax logits;')
    print('   the RMSE degradation from skipping renormalization (the no-renorm arm) is fully attributable to regional-total mismatch (the scale term),')
    print('   and the shape term is zero up to floating-point precision.')